# GridPulse BR — Gold Service Quality Product

## Objective

Build an analytics-ready Gold dataset focused on the DEC and FEC electricity
continuity indicators.

The Gold layer transforms the trusted Silver dataset into a business-oriented
data product suitable for SQL analysis, dashboards and regulatory comparison.

## Gold grain

One row represents:

- one consumer-unit set
- one year
- one reporting period

## Gold responsibilities

- Filter the analytical scope to DEC and FEC
- Pivot the indicator structure into analytics-friendly columns
- Preserve distributor and consumer-set attributes
- Validate the expected analytical grain
- Persist an analytics-ready Delta table

## Out of scope

This first Gold MVP does not yet:

- compare actual values against regulatory limits
- classify regulatory compliance
- aggregate results at distributor level
- calculate rankings or scores

These capabilities will be added incrementally.

In [0]:
from pyspark.sql import functions as F

SOURCE_TABLE = "workspace.gridpulse.silver_aneel_continuity_indicators"
TARGET_TABLE = "workspace.gridpulse.gold_service_quality"

GOLD_GRAIN = [
    "IdeConjUndConsumidoras",
    "AnoIndice",
    "NumPeriodoIndice"
]

PRODUCT_INDICATORS = ["DEC", "FEC"]

print(f"Source table       : {SOURCE_TABLE}")
print(f"Target table       : {TARGET_TABLE}")
print(f"Product indicators : {PRODUCT_INDICATORS}")
print(f"Gold grain         : {GOLD_GRAIN}")

## 1. Read Silver data

In [0]:
df_silver = spark.table(SOURCE_TABLE)

silver_row_count = df_silver.count()

print(f"Silver rows    : {silver_row_count:,}")
print(f"Silver columns : {len(df_silver.columns)}")

## 2. Filter product scope

In [0]:
df_product_scope = (
    df_silver
    .filter(F.col("SigIndicador").isin(PRODUCT_INDICATORS))
)

product_scope_count = df_product_scope.count()

print(f"DEC/FEC rows : {product_scope_count:,}")

display(
    df_product_scope
    .groupBy("SigIndicador")
    .count()
    .orderBy("SigIndicador")
)

## 3. Build analytics-ready DEC/FEC structure

Transform the long-format indicator dataset into a wide analytical structure.

Each Gold row represents one consumer-unit set for one year and reporting
period, with DEC and FEC available as separate analytical measures.

In [0]:
GOLD_DIMENSIONS = [
    "IdeConjUndConsumidoras",
    "DscConjUndConsumidoras",
    "SigAgente",
    "NumCNPJ",
    "AnoIndice",
    "NumPeriodoIndice"
]

df_gold = (
    df_product_scope
    .groupBy(*GOLD_DIMENSIONS)
    .pivot("SigIndicador", PRODUCT_INDICATORS)
    .agg(F.first("VlrIndiceEnviado"))
)

gold_row_count = df_gold.count()

print(f"Gold rows    : {gold_row_count:,}")
print(f"Gold columns : {len(df_gold.columns)}")

df_gold.printSchema()

## 4. Validate DEC/FEC completeness

In [0]:
indicator_completeness = (
    df_gold
    .agg(
        F.count("*").alias("gold_rows"),
        F.sum(F.when(F.col("DEC").isNull(), 1).otherwise(0)).alias("missing_dec"),
        F.sum(F.when(F.col("FEC").isNull(), 1).otherwise(0)).alias("missing_fec")
    )
)

display(indicator_completeness)

## 5. Validate Gold business grain

In [0]:
duplicate_gold_grain = (
    df_gold
    .groupBy(*GOLD_GRAIN)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_gold_grain_count = duplicate_gold_grain.count()

print(f"Duplicate Gold grain groups : {duplicate_gold_grain_count:,}")

assert duplicate_gold_grain_count == 0, (
    f"Gold grain validation failed: "
    f"{duplicate_gold_grain_count:,} duplicate groups found."
)

print("Gold business grain validation passed.")

## 6. Persist Gold Delta table

Persist the analytics-ready DEC/FEC dataset as a managed Delta table.

This table represents the first business-facing GridPulse BR data product.

In [0]:
df_gold = (
    df_product_scope
    .groupBy(*GOLD_DIMENSIONS)
    .pivot("SigIndicador", PRODUCT_INDICATORS)
    .agg(F.first("VlrIndiceEnviado"))
    .withColumn(
        "reference_month",
        F.make_date(
            F.col("AnoIndice").cast("int"),
            F.col("NumPeriodoIndice").cast("int"),
            F.lit(1)
        )
    )
)

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Gold table created successfully: {TARGET_TABLE}")

## 7. Validate persisted Gold product

In [0]:
df_gold_persisted = spark.table(TARGET_TABLE)

persisted_gold_count = df_gold_persisted.count()

print(f"Expected Gold rows : {gold_row_count:,}")
print(f"Actual Gold rows   : {persisted_gold_count:,}")

assert persisted_gold_count == gold_row_count, (
    f"Gold reconciliation failed: expected "
    f"{gold_row_count:,}, got {persisted_gold_count:,}"
)

print("Gold reconciliation passed.")

## 8. Gold product summary

In [0]:
gold_metrics = (
    df_gold_persisted
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("SigAgente").alias("distributors"),
        F.countDistinct("IdeConjUndConsumidoras").alias("consumer_sets"),
        F.min("reference_month").alias("min_reference_month"),
        F.max("reference_month").alias("max_reference_month"),
        F.min("DEC").alias("min_dec"),
        F.max("DEC").alias("max_dec"),
        F.min("FEC").alias("min_fec"),
        F.max("FEC").alias("max_fec")
    )
    .first()
)

print("=" * 60)
print("GRIDPULSE BR — GOLD SERVICE QUALITY PRODUCT")
print("=" * 60)

print(f"Rows              : {gold_metrics['rows']:,}")
print(f"Distributors      : {gold_metrics['distributors']:,}")
print(f"Consumer sets     : {gold_metrics['consumer_sets']:,}")
print(f"Coverage          : {gold_metrics['min_reference_month']} "
      f"to {gold_metrics['max_reference_month']}")
print(f"DEC range         : {gold_metrics['min_dec']} "
      f"to {gold_metrics['max_dec']}")
print(f"FEC range         : {gold_metrics['min_fec']} "
      f"to {gold_metrics['max_fec']}")
print(f"Target table      : {TARGET_TABLE}")

print("=" * 60)
print("STATUS: SUCCESS")
print("=" * 60)